# 05 - Callaway-Sant'Anna Staggered DiD

Estimates cohort-specific ATTs using only never-treated or not-yet-treated states as controls. Aggregates to an overall ATT that is robust to treatment effect heterogeneity across cohorts.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

FARS_FILE = "fars_state_year.parquet"
CDC_FILE  = "cdc_state_year.parquet"

# Load FARS panel
if not (DATA_DIR / FARS_FILE).exists():
    raise FileNotFoundError(
        f"{FARS_FILE} not found. Run:\n"
        "  python scripts/download_fars.py\n"
        "  python src/build_fars_panel.py"
    )
fars = pd.read_parquet(DATA_DIR / FARS_FILE)
leg  = pd.read_csv("../data/codebooks/state_legalization_dates.csv")
print(f"FARS panel: {fars.shape}  |  States: {fars['state'].nunique()}  |  Years: {sorted(fars['year'].dropna().unique()[:3])}...{sorted(fars['year'].dropna().unique()[-3:])}")

In [ ]:
from linearmodels.panel import PanelOLS

In [ ]:
primary = "total_fatalities_per_100k" if "total_fatalities_per_100k" in fars.columns else "total_fatalities"

fars_reg = fars.merge(leg[['state','retail_sales_year']], on='state', how='left')
never_treated = leg[leg['retail_sales_year'].isna()]['state'].tolist()

# Study window
MIN_YR, MAX_YR = 2010, 2022
cohorts = sorted([
    int(yr) for yr in leg['retail_sales_year'].dropna().unique()
    if MIN_YR <= int(yr) <= MAX_YR
])
print(f"Cohorts: {cohorts}")
print(f"Never-treated control states: {len(never_treated)}")

## Estimate cohort-specific ATTs (C-S approach)

For each cohort g, compare cohort g states to never-treated states only. Estimate a DiD around the cohort's treatment date.

In [ ]:
cs_results = []
for g in cohorts:
    cohort_states = leg[leg['retail_sales_year']==g]['state'].tolist()
    # Control: never-treated only (clean C-S)
    sub = fars_reg[fars_reg['state'].isin(cohort_states + never_treated)].copy()
    sub['post_g'] = ((sub['state'].isin(cohort_states)) & (sub['year'] >= g)).astype(float)
    sub['treated_g'] = sub['state'].isin(cohort_states).astype(float)

    # Pre-period: t < g; Post-period: t >= g
    # Use TWFE within this restricted sample
    try:
        idx = sub.set_index(['state','year'])
        fe = PanelOLS(idx[primary], idx[['post_g']],
                      entity_effects=True, time_effects=True).fit(
                      cov_type='clustered', cluster_entity=True)
        b   = fe.params['post_g']
        ci  = fe.conf_int().loc['post_g']
        cs_results.append({
            'cohort': g,
            'n_treated': len(cohort_states),
            'att': b,
            'ci_lo': ci['lower'],
            'ci_hi': ci['upper'],
        })
    except Exception as e:
        print(f"  Cohort {g} failed: {e}")

cs_df = pd.DataFrame(cs_results)
print("\nCallaway-Sant'Anna cohort ATTs:")
print(cs_df[['cohort','n_treated','att','ci_lo','ci_hi']].round(4).to_string(index=False))

## Aggregate ATT (simple average weighted by cohort size)

In [ ]:
if not cs_df.empty:
    weights = cs_df['n_treated'] / cs_df['n_treated'].sum()
    agg_att = (cs_df['att'] * weights).sum()
    print(f"Aggregated ATT (C-S): {agg_att:.4f}")
    print()
    print("Compare to TWFE pooled estimate (notebook 04) — divergence indicates")
    print("heterogeneous effects that TWFE was averaging over incorrectly.")

## Plot cohort-specific ATTs

In [ ]:
if not cs_df.empty:
    fig, ax = plt.subplots(figsize=(10,5))
    ax.barh(cs_df['cohort'].astype(str), cs_df['att'],
            xerr=[cs_df['att']-cs_df['ci_lo'], cs_df['ci_hi']-cs_df['att']],
            capsize=4, color='steelblue', alpha=0.75, edgecolor='white')
    ax.axvline(0, color='black', lw=0.8)
    ax.axvline(agg_att, color='red', ls='--', lw=1.5, label=f'Aggregated ATT = {agg_att:.3f}')
    ax.set_xlabel(f"ATT on {primary.replace('_',' ')}")
    ax.set_ylabel("Legalization Cohort (Year)")
    ax.set_title("Callaway-Sant'Anna: Cohort-Specific ATTs\nCannabis Legalization -> Traffic Fatalities")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "05_callaway_santanna_atts.png", bbox_inches='tight')
    plt.show()